# E044 — Atlas visuel complet

Toutes les images importantes sont affichées ici : Stage 1, Stage 2, Stage 2 avec quiet zone exacte, les 9 checkpoints gamma 500, les 9 checkpoints gamma 1000, le meilleur safe, et des vues binarisées/grille.


In [ ]:
from pathlib import Path
import json, numpy as np, cv2
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Markdown, Image
R=Path('/data/e044-multi-prompt-best-pipeline-v1')
summary=json.loads((R/'prompt-summary.json').read_text(encoding='utf-8'))
P=[s['prompt_id'] for s in summary]


In [ ]:
def show_image(path,title,width=430):
    display(Markdown('#### '+title)); display(Image(filename=str(path),width=width))

def derived_views(path):
    img=np.asarray(PILImage.open(path).convert('RGB')); gray=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)
    _,otsu=cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    adapt=cv2.adaptiveThreshold(gray,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,51,5)
    fig,axs=plt.subplots(1,3,figsize=(15,5)); axs[0].imshow(img); axs[0].set_title('RGB'); axs[1].imshow(otsu,cmap='gray'); axs[1].set_title('Otsu'); axs[2].imshow(adapt,cmap='gray'); axs[2].set_title('Adaptive')
    for ax in axs: ax.axis('off')
    plt.show()

def grid_overlay(path):
    img=np.asarray(PILImage.open(path).convert('RGB')); fig,ax=plt.subplots(figsize=(8,8)); ax.imshow(img)
    p=78; m=20
    for i in range(30):
        v=p+i*m; ax.axhline(v,linewidth=.4,alpha=.4); ax.axvline(v,linewidth=.4,alpha=.4)
    ax.set_title('Grille exacte 29×29 / module 20 / padding 78'); ax.axis('off'); plt.show()


In [ ]:
for s in summary:
    pid=s['prompt_id']; d=R/'prompts'/pid
    display(Markdown(f"# {pid} — {s['family']}"))
    show_image(d/'parent/stage1.png','Stage 1')
    show_image(d/'parent/stage2.png','Stage 2 brut')
    show_image(d/'parent/stage2-exact-qz.png','Stage 2 — quiet zone exacte')
    for gamma in (500,1000):
        display(Markdown(f"## Gamma {gamma}"))
        traj=d/'trajectories'/f'e044_{pid}_gamma{gamma:04d}_r200_i08'/'images'
        fig,axs=plt.subplots(3,3,figsize=(15,15))
        for i,ax in enumerate(axs.ravel()):
            p=traj/f'iteration-{i:03d}.png'; ax.imshow(PILImage.open(p)); ax.set_title(f'i{i}'); ax.axis('off')
        plt.tight_layout(); plt.show()
    winner=Path(s['best_image_path']); show_image(winner,f"BEST SAFE — g={int(s['best_gamma'])} i{s['best_iteration']} SSR={s['best_ssr_exact_presets']}/37",600)
    derived_views(winner); grid_overlay(winner)


## Gagnant global


In [ ]:
v=json.loads((R/'verdict.json').read_text(encoding='utf-8')); display(v); show_image(R/'pipeline/99-FINAL-QR.png','99-FINAL-QR',650)
